In [9]:
import ee
import geemap
import os
import glob
import rasterio
import matplotlib.pyplot as plt
from PIL import Image
import shutil
import math
import numpy as np
import rasterio
from skimage import exposure

In [4]:
from helpers import remove_empty_folders,delete_all_white_pngs

### Earth Engine Setup

In [16]:
# !pip install earthengine-api geemap
# ****************************************************** Enter your own api key, you will have to create an earth engine account

In [8]:
ee.Authenticate()



Enter verification code:  4/1AVMBsJgehRCb31iC8BGpsxieeIrMNVEztcMCOdLTjVeJo0TPal21WKNQSZ0



Successfully saved authorization token.


In [9]:
ee.Initialize()

#### Generating the coordinate list for the undisturbed forest class

In [ ]:
output_base = os.path.join('downloaded', 'tiff_files')
os.makedirs(output_base, exist_ok=True)


In [1]:
subtypes = ee.ImageCollection('JRC/GFC2020_subtypes/V0')
gft = subtypes.mosaic().select('Map')    # now gft is an ee.Image

# 2) Build the “primary forest” mask (class == 10)
primary_mask = gft.eq(10)

# 3) Define the region for the African rainforest
basin = ee.Geometry.Rectangle([11.0, -5.0, 30.0, 5.5])

# 4) Scatter 500 random points
random_pts = ee.FeatureCollection.randomPoints(basin, 2000)

# 5) Flag each point with the mask value under it
def flag_point(pt):
    vals = primary_mask.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=pt.geometry(),
        scale=30
    )
    return pt.set('isPrimary', vals.get('Map'))

flagged = random_pts.map(flag_point)

# 6) Filter to keep only primary‐forest points
primary_pts = flagged.filter(ee.Filter.eq('isPrimary', 1))
count = primary_pts.size().getInfo()
print("Total primary‐forest points:", count)



# Extract the first 1000 coords
coords = primary_pts.limit(1000) \
    .map(lambda f: ee.Feature(None, {
        'lon': f.geometry().coordinates().get(0),
        'lat': f.geometry().coordinates().get(1)
    })) \
    .reduceColumns(
        reducer=ee.Reducer.toList(2),
        selectors=['lon','lat']
    ) \
    .get('list') \
    .getInfo()

print("Extracted coords count:", len(coords))
# for lon, lat in coords:
#     print(f"({lat:.6f}, {lon:.6f})")

### download images

In [3]:
base_dir = os.path.join('negative_class')
os.makedirs(base_dir, exist_ok=True)


half_side = 2.5 / 111.32   # Tile half‑side (2.5 km in degrees)

Filter settings
start_date   = '2015-01-01'
end_date     = '2025-01-01'
cloud_thresh = 0.1
bands        = ['B2','B3','B4','B5']  #R,G, B and NIR bands
max_images   = 1  # per location

for lat, lon in coords:
    # Create region directory and visible subfolder
    region_name = f"{lat:.6f}_{lon:.6f}"
    vis_dir     = os.path.join(base_dir, region_name)
    os.makedirs(vis_dir, exist_ok=True)

    # Define the square ROI
    roi = ee.Geometry.Rectangle([
        lon - half_side, lat - half_side,
        lon + half_side, lat + half_side
    ])
    
    # Filter for Landsat‑8 C2 TOA collection only
    col = (
        ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
          .filterDate(start_date, end_date)
          .filterBounds(roi)
          .filter(ee.Filter.lt('CLOUD_COVER', cloud_thresh))
          .select(['B4','B3','B2'])  # only RGB bands
          .sort('CLOUD_COVER')
    )
    
    # Determine how many to download
    count = col.size().getInfo()
    n = min(max_images, count)
    print(f"{region_name}: {count} scenes found, downloading {n} visible images")
    if n == 0:
        continue
    
    # Download the first n scenes (visible only)
    img_list = col.toList(n)
    for i in range(n):
        img   = ee.Image(img_list.get(i))
        props = img.getInfo()['properties']
        ts    = props.get('system:time_start')
        date  = ee.Date(ts).format('YYYYMMdd').getInfo()

        # Visible composite (RGB)
        visible = img  # already selecting B4,B3,B2 above
        out_fp  = os.path.join(vis_dir, f"{region_name}_{date}.tif")
        geemap.ee_export_image(
            visible,
            out_fp,
            scale=30,
            region=roi,
            file_per_band=False
        )

        cloud = props.get('CLOUD_COVER')
        #print(f"  ▶ {date} (cloud {cloud}%) → saved to {region_name}")

### Processing Tiff--> Png

In [11]:



NEG_DIR = "negative_class"

for subdir in os.listdir(NEG_DIR):
    tile_dir = os.path.join(NEG_DIR, subdir)
    if not os.path.isdir(tile_dir):
        continue


    vis_dir = os.path.join(tile_dir, "visible")
    ir_dir  = os.path.join(tile_dir, "infrared")
    os.makedirs(vis_dir, exist_ok=True)
    os.makedirs(ir_dir, exist_ok=True)

    # Find the TIFF files
    tiff_paths = glob.glob(os.path.join(tile_dir, "*.tif")) + \
                 glob.glob(os.path.join(tile_dir, "*.tiff"))
    if not tiff_paths:
        print(f"No TIFF in {tile_dir}, skipping.")
        continue
    tiff_path = tiff_paths[0]
    base = os.path.splitext(os.path.basename(tiff_path))[0]

    # Read all bands as float32
    with rasterio.open(tiff_path) as src:
        data = src.read().astype(np.float32)  # (bands, H, W)

    # ─── Build 8-bit RGB ─────────────────────────────────────────────────────────────
    # Take the first 3 bands (R, G, B), clip to [0,1], then scale to [0,255]
    rgb = data[:3]  
    rgb = np.clip(rgb, 0.0, 1.0)
    rgb_u8 = (rgb * 255).astype(np.uint8)    # shape = (3, H, W)

    
    rgb_img = np.transpose(rgb_u8, (1, 2, 0)) # Transpose to (H, W, 3) & save as a true 3-channel PNG
    Image.fromarray(rgb_img, mode="RGB") \
         .save(os.path.join(vis_dir, f"{base}.png"))

    # ─── Save NIR band ────────────────────────────────────────────────────
    if data.shape[0] >= 4:
        nir = data[3]  # still float32
        np.save(os.path.join(ir_dir, f"{base}.npy"), nir)



### Some cleaning steps to remove empty folders and corrupted images

In [12]:
# Base and negative_class directory
BASE_DIR = os.getcwd()
NEG_DIR  = os.path.join(BASE_DIR, "negative_class")

# Pattern to match everything inside each “visible” folder
pattern = os.path.join(NEG_DIR, "*", "visible", "*")

# Loop and delete
for path in glob.glob(pattern):
    try:
        if os.path.isfile(path) or os.path.islink(path):
            os.remove(path)
            print(f"Deleted file: {path}")
        elif os.path.isdir(path):
            # if there happens to be a subfolder, remove its contents recursively
            for root, dirs, files in os.walk(path, topdown=False):
                for f in files:
                    fp = os.path.join(root, f)
                    os.remove(fp)
                    print(f"Deleted file: {fp}")
                for d in dirs:
                    dp = os.path.join(root, d)
                    os.rmdir(dp)
                    print(f"Deleted empty dir: {dp}")
    except Exception as e:
        print(f"Error deleting {path}: {e}")

In [ ]:
base_directory = "negative_class"

remove_empty_folders(base_directory)
delete_all_white_pngs(base_directory)

#### Copy Visible Pngs to the undisturbed forest class folder. This is necessary to transition from a per-location to per-class directory structure

In [2]:
BASE_DIR = os.getcwd()

NEG_DIR = os.path.join(BASE_DIR, "negative_class")
UND_DIR = os.path.join(BASE_DIR, "undisturbed_forest")
os.makedirs(UND_DIR, exist_ok=True)

# Find all visible PNGs in the negative_class subfolders
pattern = os.path.join(NEG_DIR, "*", "visible", "*.png")

for vis_path in glob.glob(pattern):
    filename = os.path.basename(vis_path)
    dest_path = os.path.join(UND_DIR, filename)
    shutil.copy2(vis_path, dest_path)
    #print(f"Copied {vis_path} → {dest_path}")